# Radiation test data

- Download required MSG and ERA5 data from a product list
- Add radiation data from MSG to the dataset
- Add ozone and temperature data from era5 to the dataset
- Write files needed as input and output for test (except CAMS and water column which is already available in S2/Maja product) : albedo, rsd, rld, fdiffuse, ozone et température

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import datetime as dt
import os

import rasterio as rio
from sensorsio import utils

In [ ]:
from etdataset import dem, era5, msg
from etdataset.reader import get_product_reader
from etdataset.writer import write_dataset

# Logging

In [ ]:
import logging

from etdataset.logging import LoggerManager

LoggerManager.set_level(logging.INFO)

## Inputs / Outputs

In [ ]:
S2_dir = "S2"  # to be modified to point on the directory containing S2 files

In [ ]:
DEM_dir = "dem"  # to be modified to point on the directory containing DEM files

In [ ]:
data_dir = "data_aux"  # to be modified to point on the directory containing auxiliary data with sub-directories named MSG_data, ERA5_data,

In [ ]:
product_list = os.listdir(S2_dir)

In [ ]:
product_list

In [ ]:
output_dir = "out"  # to be modified to point on the output directory

## Select data

In [ ]:
product_index = 0  # to be modified to point the data product to process

In [ ]:
product_path = os.path.join(S2_dir, product_list[product_index])

In [ ]:
product_path

# Read Data

In [ ]:
reader = get_product_reader(product_path=product_path)

In [ ]:
s2_dataset = reader.read_vis_bands()

In [ ]:
date = dt.datetime(2024, 10, 21, 10, 33)

# Add DEM

In [ ]:
s2_dataset = dem.add_dem(data=s2_dataset, mnt_dir=DEM_dir)

In [ ]:
s2_dataset

# Manage CRS and date

In [ ]:
crs = s2_dataset.rio.crs
bounds = rio.coords.BoundingBox(*s2_dataset.rio.bounds())

In [ ]:
s2_dataset.attrs

In [ ]:
s2_dataset.data_vars

In [ ]:
latlon_bounds = utils.bb_transform(
    source_crs=str(crs), target_crs="EPSG:4326", bounding_box=bounds
)

In [ ]:
date = dt.datetime.combine(
    s2_dataset.attrs["vis_date"], s2_dataset.attrs["vis_time"]
)

# Add MSG data

In [ ]:
msg.download(date=date, latlon_bbox=latlon_bounds, path=data_dir)

In [ ]:
s2_dataset = msg.add(data=s2_dataset, path=data_dir)

In [ ]:
s2_dataset

# Add era5 data

In [ ]:
era5.download(date=date, dataset=era5.ERA5Dataset.ERA5, path=data_dir)

In [ ]:
s2_dataset = era5.add(
    data=s2_dataset,
    dataset=era5.ERA5Dataset.ERA5,
    variables=[
        era5.ERA5Var.TEMPERATURE,
        era5.ERA5Var.DEWPOINT_TEMPERATURE,
        era5.ERA5Var.TOTAL_COLUMN_OZONE,
    ],
    path=data_dir,
)

In [ ]:
s2_dataset

# Write data

In [ ]:
write_dataset(xrds=s2_dataset, bands=None, directory=output_dir, separate=True)

# Visualisations

In [ ]:
s2_dataset.fdiff_msg.plot()

In [ ]:
s2_dataset.rsd_msg.plot()

In [ ]:
s2_dataset.rld_msg.plot()

In [ ]:
s2_dataset.albedo.plot()